# STAC integration

This tutorial shows how to build a lazy `RasterData` from a **STAC**
catalogue with `RasterData.from_stac`, using the
[EcoDataCube](https://stac.opengeohub.org/v1/cat/ecodatacube) catalogue
(67 European variables, EPSG:3035, 30 m).  The result holds remote COG
hrefs + dates only, so we sample it with `SpaceTimeOverlay` (point
queries) and read a small tile — never the full continental grid.


In [1]:
import os

os.environ["PROJ_LIB"] = "/skmap/lib/python3.12/site-packages/rasterio/proj_data"
os.environ["PROJ_DATA"] = "/skmap/lib/python3.12/site-packages/rasterio/proj_data"


In [2]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from skmap.io import RasterData
from skmap.data import toy
from skmap.overlay import SpaceTimeOverlay


## Loading layers from the STAC catalogue

`from_stac` queries the per-collection `/items` endpoint and returns a
**lazy** `RasterData`: one `info` row per data asset, with `collection`,
`asset`, `year`, `gsd` and `epsg` columns.  The `datetime` window is
filtered client-side.


In [3]:
?RasterData.from_stac

Signature:
RasterData.from_stac(
    url: str,
    collections: Union[str, List[str]],
    datetime: str = None,
    bbox: List[float] = None,
    bands: List[str] = None,
    max_items: int = None,
    limit: int = 500,
    date_format: str = '%Y%m%d',
    ignore_29feb: bool = True,
    backend: Union[str, ForwardRef('ComputeBackend')] = 'numpy',
    verbose: bool = False,
)
Docstring:
Build a lazy RasterData from a STAC catalogue (classmethod).

Queries the per-collection ``/items`` endpoint and yields one ``info``
row per data asset (``roles`` contains ``"data"``).  The result holds
remote COG hrefs + dates only (no ``.read()``); its ``info`` carries
``collection``, ``asset``, ``year``, ``gsd`` and ``epsg`` columns.
See :class:`skmap.io.sources.StacSource` for the query rules.
File:      ~/scikit-map/skmap/io/base.py
Type:      method

In [4]:
rdata = RasterData.from_stac(
    url="https://stac.opengeohub.org/v1/cat/landmetric",
    collections=[
        "ndvi_glad.swa.ard2_go_landmetric",
        "ndwi_glad.swa.ard2_go_landmetric"
    ],
    datetime="2015-01-01/2024-12-31",
    backend='cpp'
)
rdata


In [5]:
rdata.info[["group", "name", "start_date", "end_date", "collection", "asset", "year", "epsg", "gsd"]]


,group,name,start_date,end_date,collection,asset,year,epsg,gsd
0,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s_1101,2024-11-01,2024-12-31,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s,2024,4326,30.0
1,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s_0901,2024-09-01,2024-10-31,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s,2024,4326,30.0
2,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s_0701,2024-07-01,2024-08-31,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s,2024,4326,30.0
3,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s_0501,2024-05-01,2024-06-30,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s,2024,4326,30.0
4,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s_0301,2024-03-01,2024-04-30,ndvi_glad.swa.ard2_go_landmetric,ndvi_glad.swa.ard2_m_30m_s,2024,4326,30.0
...,...,...,...,...,...,...,...,...,...
109,ndwi_glad.swa.ard2_go_landmetric,ndwi_glad.swa.ard2_m_30m_s_0901,2015-09-01,2015-10-31,ndwi_glad.swa.ard2_go_landmetric,ndwi_glad.swa.ard2_m_30m_s,2015,4326,30.0
110,ndwi_glad.swa.ard2_go_landmetric,ndwi_glad.swa.ard2_m_30m_s_0701,2015-07-01,2015-08-31,ndwi_glad.swa.ard2_go_landmetric,ndwi_glad.swa.ard2_m_30m_s,2015,4326,30.0
111,ndwi_glad.swa.ard2_go_landmetric,ndwi_glad.swa.ard2_m_30m_s_0501,2015-05-01,2015-06-30,ndwi_glad.swa.ard2_go_landmetric,ndwi_glad.swa.ard2_m_30m_s,2015,4326,30.0
112,ndwi_glad.swa.ard2_go_landmetric,ndwi_glad.swa.ard2_m_30m_s_0301,2015-03-01,2015-04-30,ndwi_glad.swa.ard2_go_landmetric,ndwi_glad.swa.ard2_m_30m_s,2015,4326,30.0


In [7]:
rdata.get_groups()


['ndvi_glad.swa.ard2_go_landmetric', 'ndwi_glad.swa.ard2_go_landmetric']

### Multi-column runners on a STAC catalogue

The `info` frame carries one column per STAC variable (`collection`,
`asset`, `year`, `gsd`, `epsg`), so runners are not limited to the
predefined `group` column:

* **`select(...)`** filters layers by any variable column, e.g.
  `rdata.select(asset="red", year=2019)`;
* **`match_col=`** tells a runner (e.g.
  `process.NormalizedDifference("red", "nir")`) to resolve its input
  bands from a variable column instead of `name`;
* **`by=`** computes **one output band per partition** — e.g.
  `by="year"` produces a derived band for every year, with the partition
  values stored back into the new `info` rows.

```python
rdata.run(
    process.NormalizedDifference("red", "nir"),
    match_col="asset",   # "red"/"nir" live in the asset column
    by="year",           # one NDVI band per year
)
```

## Space-time overlay with toy reference samples

The bundled `toy.lc_samples()` are already in **EPSG:3035** — the same CRS
as the ecodatacube layers — so we can overlay them directly, sampling the
remote COGs at the point locations without loading whole rasters.


In [8]:
samples = toy.lc_samples()

overlay = SpaceTimeOverlay(
    points=samples,
    col_date="date",
    rasterdata=rdata,
    raster_tiles=None,
    verbose=False,
)
train = overlay.run(max_ram_mb=512, out_file_name=None)


Dropping 0 points out of 41 because out of extent
Dropping 0 points out of 37 because out of extent
Dropping 0 points out of 40 because out of extent
Dropping 0 points out of 43 because out of extent
Dropping 0 points out of 38 because out of extent
Dropping 0 points out of 57 because out of extent


In [14]:
train.head()


,date,label,code,target,lon,lat,ndvi_glad.swa.ard2_m_30m_s_1101,ndvi_glad.swa.ard2_m_30m_s_0901,ndvi_glad.swa.ard2_m_30m_s_0701,ndvi_glad.swa.ard2_m_30m_s_0501,ndvi_glad.swa.ard2_m_30m_s_0301,ndvi_glad.swa.ard2_m_30m_s_0101,ndwi_glad.swa.ard2_m_30m_s_1101,ndwi_glad.swa.ard2_m_30m_s_0901,ndwi_glad.swa.ard2_m_30m_s_0701,ndwi_glad.swa.ard2_m_30m_s_0501,ndwi_glad.swa.ard2_m_30m_s_0301,ndwi_glad.swa.ard2_m_30m_s_0101
0,2015-01-01,Pastures,231,18,4.024372e+06,3.214363e+06,199.0,196.0,154.0,175.0,185.0,179.0,138.0,143.0,138.0,159.0,157.0,129.0
1,2015-01-01,Pastures,231,18,4.020732e+06,3.212050e+06,169.0,195.0,189.0,185.0,200.0,180.0,122.0,152.0,149.0,154.0,154.0,130.0
2,2015-01-01,Broad-leaved forest,311,23,4.026238e+06,3.215984e+06,202.0,218.0,223.0,229.0,214.0,210.0,131.0,163.0,166.0,176.0,146.0,136.0
3,2015-01-01,Broad-leaved forest,311,23,4.027110e+06,3.216442e+06,209.0,217.0,223.0,228.0,223.0,215.0,136.0,152.0,167.0,172.0,150.0,153.0
4,2015-01-01,Water bodies,512,41,4.023718e+06,3.213876e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Reading a small tile and plotting

`read(extent=...)` reads only a bounding box (here the samples' AOI in
EPSG:3035), and `plot` renders each bimonthly composite with a
`{field}` title template.


In [8]:
from skmap.io.base import _read_rasters_cpp

In [6]:
base.

<function skmap.io.base._read_rasters_cpp(raster_files, band, window, n_jobs, dtype, gdal_opts, verbose, overview=None)>

In [17]:
tile = rdata.read(extent=[4020674, 3210142, 4028246, 3217730], extent_epsg=3035)
tile.plot(cmap="RdYlGn", img_title_text="{start_date:%Y-%m-%d}")


ERROR 1: PROJ: internal_proj_create_from_database: /skmap/lib/python3.12/site-packages/pyproj/proj_dir/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 4 whereas a number >= 6 is expected. It comes from another PROJ installation.


CRSError: The EPSG code is unknown. PROJ: internal_proj_create_from_database: /skmap/lib/python3.12/site-packages/pyproj/proj_dir/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 4 whereas a number >= 6 is expected. It comes from another PROJ installation.